# MTG Premodern - Actualización Incremental

Este notebook actualiza la base de datos con torneos y mazos nuevos.

**Flujo:**
1. Subir `premodern.db` existente
2. Scrapear solo páginas nuevas de `results.php`
3. Crear/actualizar vistas
4. Scrapear cartas de mazos pendientes (mismo método que notebook 02: `requests` puro)
5. Descargar DB actualizada


In [ ]:
# Celda 1: Instalar dependencias
!pip install -q requests beautifulsoup4 lxml tqdm


In [ ]:
# Celda 2: Subir DB existente

import os

DB_FILE = "premodern.db"

if not os.path.exists(DB_FILE):
    try:
        from google.colab import files
        print("Subí el archivo premodern.db:")
        uploaded = files.upload()
    except ImportError:
        print(f"ERROR: No se encontró {DB_FILE}")
else:
    print(f"{DB_FILE} encontrado ({os.path.getsize(DB_FILE) / 1024 / 1024:.1f} MB)")

In [ ]:
# Celda 3: Imports y estado actual

import sqlite3
import time
import re
import logging
from datetime import datetime
from urllib.parse import parse_qs, urlparse

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

logging.basicConfig(level=logging.WARNING, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

BASE_URL    = "https://www.tcdecks.net"
RESULTS_URL = f"{BASE_URL}/results.php"
DECK_URL    = f"{BASE_URL}/deck.php"
DELAY           = 1.5    # segundos entre requests de cartas
DELAY_META      = 1.5    # segundos entre requests de metadata
BATCH_SIZE      = 200    # commit cada N mazos
MAX_CONSECUTIVE_BLOCKS = 10

conn = sqlite3.connect(DB_FILE)
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA foreign_keys=ON")

# Agregar columnas a cards si la DB viene de una versión anterior
for col, typedef in [("card_type", "TEXT"), ("cardmarket_id", "INTEGER"), ("cardmarket_url", "TEXT")]:
    try:
        conn.execute(f"ALTER TABLE cards ADD COLUMN {col} {typedef}")
        conn.commit()
        print(f"Columna '{col}' agregada a cards.")
    except Exception:
        pass

total_tournaments = conn.execute("SELECT COUNT(*) FROM tournaments").fetchone()[0]
total_decks       = conn.execute("SELECT COUNT(*) FROM decks").fetchone()[0]
max_date          = conn.execute("SELECT MAX(date) FROM tournaments").fetchone()[0]
unscraped_cards   = conn.execute("SELECT COUNT(*) FROM decks WHERE cards_scraped = 0").fetchone()[0]

print(f"Estado actual de la DB:")
print(f"  Torneos:          {total_tournaments:,}")
print(f"  Mazos:            {total_decks:,}")
print(f"  Última fecha:     {max_date}")
print(f"  Mazos sin cartas: {unscraped_cards:,}")


In [ ]:
# Celda 4: Funciones utilitarias (mismas que notebook 01)

def parse_date(date_str):
    date_str = date_str.strip()
    for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(date_str, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    raise ValueError(f"Cannot parse date: {date_str}")

def parse_position(text):
    match = re.search(r"(\d+)\s+of\s+(\d+)", text)
    return (int(match.group(1)), int(match.group(2))) if match else (None, None)

def extract_deck_ids(href):
    parsed = urlparse(href)
    params = parse_qs(parsed.query)
    tid = int(params["id"][0]) if "id" in params else None
    did = int(params["iddeck"][0]) if "iddeck" in params else None
    return tid, did

def detect_source(name):
    lower = name.lower()
    if any(kw in lower for kw in ("mtgo", "league", "challenge", "preliminary", "showcase")):
        return "mol"
    if "webcam" in lower:
        return "webcam"
    return "paper"

BASE_SITE = "https://www.tcdecks.net"

# Setup sesión HTTP
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
})
retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retry))

print("Listo.")

In [ ]:
# Celda 5: Scraping incremental de metadata
#
# Estructura HTML real: <tr> sin clase, 6 celdas con data-th:
#   [0] Archetype (link con iddeck), [1] Format, [2] Player,
#   [3] Tournament Name, [4] Position, [5] Date

existing_deck_ids = set(
    row[0] for row in conn.execute("SELECT id FROM decks").fetchall()
)

new_decks = 0
page = 1
consecutive_known_pages = 0
STOP_AFTER_KNOWN = 3  # Parar después de 3 páginas consecutivas sin datos nuevos

print("Scraping incremental de metadata...")
print(f"Decks conocidos: {len(existing_deck_ids):,}")

while consecutive_known_pages < STOP_AFTER_KNOWN:
    try:
        response = session.get(RESULTS_URL, params={
            "format": "Premodern", "src": "all", "page": page
        }, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "lxml")
        rows = soup.find_all("tr")

        page_new = 0
        rows_found = 0
        for row in rows:
            try:
                cells = row.find_all("td")
                if len(cells) < 6:
                    continue
                if not cells[0].get("data-th"):
                    continue

                arch_link = cells[0].find("a", href=True)
                if not arch_link:
                    continue
                archetype = arch_link.get_text(strip=True)
                href = arch_link.get("href", "")
                tournament_id, deck_id = extract_deck_ids(href)
                if not tournament_id or not deck_id:
                    continue

                rows_found += 1

                player_link = cells[2].find("a")
                player_name = player_link.get_text(strip=True) if player_link else "Unknown"
                tourn_link = cells[3].find("a")
                tournament_name = tourn_link.get_text(strip=True) if tourn_link else "Unknown"
                position_text = cells[4].get_text(strip=True)
                position, total_players = parse_position(position_text)
                date_text = cells[5].get_text(strip=True)
                if not date_text:
                    continue
                parsed_date = parse_date(date_text)
                source = detect_source(tournament_name)

                # Construir URLs
                tournament_url = f"{BASE_SITE}/deck.php?id={tournament_id}"
                archetype_url = f"{BASE_SITE}/archetype.php?archetype={archetype}&format=Premodern"

                conn.execute(
                    """INSERT INTO tournaments (id, name, date, player_count, source, url)
                       VALUES (?, ?, ?, ?, ?, ?)
                       ON CONFLICT(id) DO UPDATE SET
                         player_count = MAX(tournaments.player_count, excluded.player_count),
                         source = CASE WHEN tournaments.source = 'unknown'
                                       THEN excluded.source ELSE tournaments.source END,
                         url = COALESCE(tournaments.url, excluded.url)""",
                    (tournament_id, tournament_name, parsed_date, total_players, source, tournament_url),
                )

                if deck_id not in existing_deck_ids:
                    try:
                        conn.execute(
                            """INSERT OR IGNORE INTO decks
                               (id, tournament_id, player_name, archetype, archetype_url,
                                position, total_players, date)
                               VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
                            (deck_id, tournament_id, player_name, archetype, archetype_url,
                             position, total_players, parsed_date),
                        )
                        if conn.execute("SELECT changes()").fetchone()[0] > 0:
                            existing_deck_ids.add(deck_id)
                            page_new += 1
                            new_decks += 1
                    except sqlite3.IntegrityError:
                        pass
            except Exception as e:
                continue

        conn.commit()

        if rows_found == 0:
            break

        if page_new == 0:
            consecutive_known_pages += 1
        else:
            consecutive_known_pages = 0

        print(f"  Página {page}: {page_new} nuevos ({rows_found} filas) (total nuevos: {new_decks})")
        page += 1
        time.sleep(DELAY_META)

    except Exception as e:
        logger.error(f"Error página {page}: {e}")
        time.sleep(5)
        page += 1

print(f"\nMetadata incremental: {new_decks} mazos nuevos en {page-1} páginas.")

In [ ]:
# Celda 6: Crear/actualizar vistas
#
# Idempotente: DROP VIEW IF EXISTS antes de cada CREATE VIEW.

views = {
    "v_meta_share": """
        SELECT
            d.archetype,
            strftime('%Y-%m', d.date) AS month,
            COUNT(*) AS deck_count,
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (
                PARTITION BY strftime('%Y-%m', d.date)
            ) AS meta_share_pct
        FROM decks d
        GROUP BY d.archetype, strftime('%Y-%m', d.date)
    """,

    "v_archetype_success": """
        SELECT
            d.archetype,
            COUNT(*) AS total_entries,
            AVG(CASE WHEN d.position <= 8  THEN 1.0 ELSE 0.0 END) AS top8_rate,
            AVG(CASE WHEN d.position <= 4  THEN 1.0 ELSE 0.0 END) AS top4_rate,
            AVG(CASE WHEN d.position  = 1  THEN 1.0 ELSE 0.0 END) AS win_rate,
            AVG(d.position * 1.0 / d.total_players) AS avg_relative_position
        FROM decks d
        WHERE d.total_players >= 8
        GROUP BY d.archetype
        HAVING COUNT(*) >= 10
    """,

    "v_card_popularity": """
        SELECT
            dc.card_name,
            dc.is_sideboard,
            COUNT(DISTINCT dc.deck_id) AS deck_count,
            SUM(dc.quantity)           AS total_copies,
            AVG(dc.quantity)           AS avg_copies_per_deck
        FROM deck_cards dc
        GROUP BY dc.card_name, dc.is_sideboard
    """,

    "v_player_stats": """
        SELECT
            d.player_name,
            COUNT(*)                                              AS total_entradas,
            COUNT(DISTINCT d.tournament_id)                       AS torneos_jugados,
            COUNT(DISTINCT d.archetype)                           AS arquetipos_distintos,
            SUM(CASE WHEN d.position = 1   THEN 1 ELSE 0 END)    AS victorias,
            SUM(CASE WHEN d.position <= 4  THEN 1 ELSE 0 END)    AS top4s,
            SUM(CASE WHEN d.position <= 8  THEN 1 ELSE 0 END)    AS top8s,
            ROUND(AVG(d.position * 1.0 / d.total_players), 4)    AS avg_posicion_relativa,
            ROUND(
                SUM(CASE WHEN d.position <= 8 THEN 1.0 ELSE 0.0 END) / COUNT(*), 4
            )                                                     AS top8_rate,
            MIN(d.date)                                           AS primer_torneo,
            MAX(d.date)                                           AS ultimo_torneo
        FROM decks d
        WHERE d.total_players >= 8
        GROUP BY d.player_name
        HAVING COUNT(*) >= 3
    """,
}

for name, body in views.items():
    conn.execute(f"DROP VIEW IF EXISTS {name}")
    conn.execute(f"CREATE VIEW {name} AS {body}")

conn.commit()

existing = [
    r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='view' ORDER BY name"
    )
]
print(f"Vistas OK: {existing}")


In [ ]:
# Celda 7: Scrapear cartas de mazos pendientes
#
# Mismo método que notebook 02: requests puro con cookie verificado=1.
# Resumible: re-ejecutar retoma desde cards_scraped=0.

TYPE_MAP = {
    "Creatures": "Creature", "Instants": "Instant", "Sorceries": "Sorcery",
    "Artifacts": "Artifact", "Enchantments": "Enchantment",
    "Planeswalkers": "Planeswalker", "Land": "Land", "Lands": "Land", "Battle": "Battle",
}
CARD_RE = re.compile(r'(\d+)\s*<a\b[^>]*?href="([^"]+)"[^>]*?name="([^"]+)"[^>]*?>')


def create_session():
    s = requests.Session()
    s.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
        ),
        "Accept": (
            "text/html,application/xhtml+xml,application/xml;"
            "q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8"
        ),
        "Accept-Language":  "en-US,en;q=0.9,es;q=0.8",
        "Accept-Encoding":  "gzip, deflate, br",  # ← SIN ESTO EL SITIO BLOQUEA
        "Connection":       "keep-alive",
        "Sec-Fetch-Dest":   "document",
        "Sec-Fetch-Mode":   "navigate",
        "Sec-Fetch-Site":   "same-origin",
        "Sec-Fetch-User":   "?1",
        "Upgrade-Insecure-Requests": "1",
        "Referer": f"{BASE_URL}/results.php?format=Premodern&src=all&page=1",
    })
    retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.cookies.set("verificado", "1", domain="www.tcdecks.net", path="/")
    s.get(f"{BASE_URL}/results.php?format=Premodern&src=all&page=1", timeout=30)
    return s


def parse_deck_html(html, deck_id):
    if "Acceso Denegado" in html or len(html) < 500:
        return None
    soup = BeautifulSoup(html, "lxml")
    deck_table = soup.find("table", class_="table_deck")
    if not deck_table:
        return None
    data_cells = deck_table.find_all("td", valign="top")
    if not data_cells:
        return None

    sideboard_idx = len(data_cells) - 1 if len(data_cells) >= 2 else None
    cards = []

    for cell_idx, cell in enumerate(data_cells):
        is_sideboard = (cell_idx == sideboard_idx)
        parts = re.split(r'(<h6>[^<]*</h6>)', str(cell))
        current_type = None

        for part in parts:
            h6 = re.match(r'<h6>([^\[<]+?)(?:\s*\[\d+\])?\s*</h6>', part.strip())
            if h6:
                current_type = TYPE_MAP.get(h6.group(1).strip(), h6.group(1).strip())
                continue
            for m in CARD_RE.finditer(part):
                href = m.group(2)
                cm_id_m = re.search(r'idProduct=(\d+)', href)
                cards.append({
                    "deck_id":        deck_id,
                    "card_name":      m.group(3).strip(),
                    "quantity":       int(m.group(1)),
                    "is_sideboard":   is_sideboard,
                    "card_type":      current_type,
                    "cardmarket_id":  int(cm_id_m.group(1)) if cm_id_m else None,
                    "cardmarket_url": re.split(r'&utm_', href)[0] if href else None,
                })

    return cards if cards else None


pending = conn.execute(
    "SELECT id, tournament_id FROM decks WHERE cards_scraped = 0"
).fetchall()

print(f"Mazos pendientes: {len(pending):,}")
print(f"Tiempo estimado:  ~{len(pending) * DELAY / 3600:.1f} horas")

if pending:
    session = create_session()
    print(f"Sesión iniciada. Cookies: {list(session.cookies.keys())}")

    success = 0
    blocked = 0
    errors  = 0
    consecutive_blocks = 0

    pbar = tqdm(pending, desc="Cartas", unit="mazo")

    for deck_id, tourn_id in pbar:
        try:
            r = session.get(f"{DECK_URL}?id={tourn_id}&iddeck={deck_id}", timeout=30)
            cards = parse_deck_html(r.text, deck_id)

            if cards is None:
                blocked += 1
                consecutive_blocks += 1
                if consecutive_blocks >= MAX_CONSECUTIVE_BLOCKS:
                    print(f"\n⚠️ {MAX_CONSECUTIVE_BLOCKS} bloqueos seguidos — reiniciando sesión...")
                    time.sleep(30)
                    session = create_session()
                    consecutive_blocks = 0
                time.sleep(DELAY)
                continue

            consecutive_blocks = 0

            for c in cards:
                conn.execute(
                    """INSERT INTO cards (name, card_type, cardmarket_id, cardmarket_url)
                       VALUES (?, ?, ?, ?)
                       ON CONFLICT(name) DO UPDATE SET
                         card_type      = COALESCE(cards.card_type,      excluded.card_type),
                         cardmarket_id  = COALESCE(cards.cardmarket_id,  excluded.cardmarket_id),
                         cardmarket_url = COALESCE(cards.cardmarket_url, excluded.cardmarket_url)""",
                    (c["card_name"], c["card_type"], c["cardmarket_id"], c["cardmarket_url"]),
                )
                conn.execute(
                    """INSERT OR REPLACE INTO deck_cards
                       (deck_id, card_name, quantity, is_sideboard)
                       VALUES (?, ?, ?, ?)""",
                    (c["deck_id"], c["card_name"], c["quantity"], int(c["is_sideboard"])),
                )

            conn.execute("UPDATE decks SET cards_scraped = 1 WHERE id = ?", (deck_id,))
            success += 1

            if success % BATCH_SIZE == 0:
                conn.commit()

        except Exception as e:
            errors += 1
            if errors % 100 == 0:
                logger.warning(f"Error #{errors} deck {deck_id}: {e}")

        pbar.set_postfix({"ok": success, "block": blocked, "err": errors})
        time.sleep(DELAY)

    conn.commit()
    print(f"\n{'='*50}")
    print(f"Éxitos: {success:,}  |  Bloqueados: {blocked:,}  |  Errores: {errors:,}")
else:
    print("No hay mazos pendientes de cartas.")


In [ ]:
# Celda 7b: Funciones para sincronizar datos de Scryfall (mismo metodo que notebook 02)
# Funciones para sincronizar datos de Scryfall (metadata completa + precios)
import json as _json_scryfall
import requests as _requests_scryfall

SCRYFALL_COLLECTION_URL = "https://api.scryfall.com/cards/collection"
SCRYFALL_BATCH_SIZE = 75
SCRYFALL_DELAY = 0.25  # 4 req/seg — conservador para evitar rate limit de Scryfall

SCRYFALL_NEW_COLUMNS = [
    ("mana_cost", "TEXT"), ("cmc", "REAL"), ("type_line", "TEXT"),
    ("oracle_text", "TEXT"), ("flavor_text", "TEXT"), ("power", "TEXT"),
    ("toughness", "TEXT"), ("loyalty", "TEXT"), ("colors", "TEXT"),
    ("color_identity", "TEXT"), ("keywords", "TEXT"), ("produced_mana", "TEXT"),
    ("rarity", "TEXT"), ("set_code", "TEXT"), ("set_name", "TEXT"),
    ("released_at", "TEXT"), ("collector_number", "TEXT"), ("layout", "TEXT"),
    ("image_uri", "TEXT"), ("scryfall_id", "TEXT"), ("price_usd", "REAL"),
    ("price_updated_at", "TIMESTAMP"), ("scryfall_raw", "TEXT"),
    ("scryfall_synced_at", "TIMESTAMP"),
]

def ensure_scryfall_schema(conn):
    existing = {row[1] for row in conn.execute("PRAGMA table_info(cards)").fetchall()}
    for col, typedef in SCRYFALL_NEW_COLUMNS:
        if col not in existing:
            conn.execute(f"ALTER TABLE cards ADD COLUMN {col} {typedef}")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS card_price_history (
            card_name TEXT NOT NULL REFERENCES cards(name),
            price_usd REAL,
            fetched_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (card_name, fetched_at)
        )
    """)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_price_history_card ON card_price_history(card_name)")
    conn.commit()

def cheapest_usd_price(prices):
    """Precio mas economico entre las versiones devueltas por Scryfall (normal/foil/etched), solo USD."""
    candidates = [prices.get("usd"), prices.get("usd_foil"), prices.get("usd_etched")]
    nums = [float(p) for p in candidates if p is not None]
    return min(nums) if nums else None

def _join_list(values):
    return ",".join(values) if values else ""

def fetch_scryfall_batch(card_names):
    """Consulta Scryfall en lotes de 75 nombres. Devuelve (dict nombre->card_dict, not_found)."""
    results = {}
    not_found = []
    for i in range(0, len(card_names), SCRYFALL_BATCH_SIZE):
        batch = card_names[i:i + SCRYFALL_BATCH_SIZE]
        identifiers = [{"name": n} for n in batch]
        resp = _requests_scryfall.post(
            SCRYFALL_COLLECTION_URL, json={"identifiers": identifiers}, timeout=30
        )
        resp.raise_for_status()
        data = resp.json()
        for card in data.get("data", []):
            results[card.get("name")] = card
        for nf in data.get("not_found", []):
            not_found.append(nf.get("name"))
        time.sleep(SCRYFALL_DELAY)
    return results, not_found

def sync_scryfall_metadata(conn):
    """Trae metadata completa (tipo, texto, colores, precio, imagen, etc.) para cartas nunca sincronizadas."""
    ensure_scryfall_schema(conn)
    pending = [r[0] for r in conn.execute("SELECT name FROM cards WHERE scryfall_id IS NULL").fetchall()]
    print(f"[Scryfall metadata] Cartas pendientes: {len(pending):,}")
    if not pending:
        print("[Scryfall metadata] Nada que actualizar.")
        return

    results, not_found = fetch_scryfall_batch(pending)

    for name, card in results.items():
        mana_cost = card.get("mana_cost", "") or ""
        if not mana_cost and "card_faces" in card:
            faces = card["card_faces"]
            mana_cost = "//".join(f.get("mana_cost", "") or "" for f in faces)
        price = cheapest_usd_price(card.get("prices", {}) or {})

        conn.execute(
            """UPDATE cards SET
                 mana_cost = ?, cmc = ?, type_line = ?, oracle_text = ?, flavor_text = ?,
                 power = ?, toughness = ?, loyalty = ?, colors = ?, color_identity = ?,
                 keywords = ?, produced_mana = ?, rarity = ?, set_code = ?, set_name = ?,
                 released_at = ?, collector_number = ?, layout = ?, image_uri = ?,
                 scryfall_id = ?, price_usd = ?, price_updated_at = CURRENT_TIMESTAMP,
                 scryfall_raw = ?, scryfall_synced_at = CURRENT_TIMESTAMP
               WHERE name = ?""",
            (mana_cost, card.get("cmc"), card.get("type_line"), card.get("oracle_text"),
             card.get("flavor_text"), card.get("power"), card.get("toughness"), card.get("loyalty"),
             _join_list(card.get("colors")), _join_list(card.get("color_identity")),
             _join_list(card.get("keywords")), _join_list(card.get("produced_mana")),
             card.get("rarity"), card.get("set"), card.get("set_name"), card.get("released_at"),
             card.get("collector_number"), card.get("layout"),
             (card.get("image_uris") or {}).get("normal"), card.get("id"), price,
             _json_scryfall.dumps(card), name),
        )
        if price is not None:
            conn.execute(
                "INSERT OR REPLACE INTO card_price_history (card_name, price_usd, fetched_at) "
                "VALUES (?, ?, CURRENT_TIMESTAMP)",
                (name, price),
            )
    conn.commit()
    print(f"[Scryfall metadata] Actualizadas: {len(results):,}")
    print(f"[Scryfall metadata] No encontradas: {len(not_found)}")
    if not_found:
        print("[Scryfall metadata] Ejemplos no encontrados:", not_found[:20])

def refresh_scryfall_prices(conn):
    """Refresca el precio mas economico de TODAS las cartas ya sincronizadas y
    guarda una fila nueva en card_price_history (para trackear evolucion en el tiempo)."""
    ensure_scryfall_schema(conn)
    synced = [r[0] for r in conn.execute("SELECT name FROM cards WHERE scryfall_id IS NOT NULL").fetchall()]
    print(f"[Scryfall precios] Cartas a refrescar: {len(synced):,}")
    if not synced:
        print("[Scryfall precios] Nada que refrescar (corre sync_scryfall_metadata primero).")
        return

    results, not_found = fetch_scryfall_batch(synced)
    updated = 0
    for name, card in results.items():
        price = cheapest_usd_price(card.get("prices", {}) or {})
        if price is None:
            continue
        conn.execute(
            "UPDATE cards SET price_usd = ?, price_updated_at = CURRENT_TIMESTAMP WHERE name = ?",
            (price, name),
        )
        conn.execute(
            "INSERT OR REPLACE INTO card_price_history (card_name, price_usd, fetched_at) "
            "VALUES (?, ?, CURRENT_TIMESTAMP)",
            (name, price),
        )
        updated += 1
    conn.commit()
    print(f"[Scryfall precios] Precios actualizados: {updated:,}")
    if not_found:
        print(f"[Scryfall precios] No encontradas: {len(not_found)}")

In [ ]:
# Celda 7c: Sincronizar metadata de cartas nuevas + refrescar precios
# Sincronizar metadata de cartas nuevas + refrescar precios de todas (idempotente)
sync_scryfall_metadata(conn)
refresh_scryfall_prices(conn)


In [ ]:
# Celda 8: Resumen y descarga

total_t = conn.execute("SELECT COUNT(*) FROM tournaments").fetchone()[0]
total_d = conn.execute("SELECT COUNT(*) FROM decks").fetchone()[0]
total_c = conn.execute("SELECT COUNT(*) FROM cards").fetchone()[0]
scraped = conn.execute("SELECT COUNT(*) FROM decks WHERE cards_scraped = 1").fetchone()[0]

print(f"Estado final:")
print(f"  Torneos: {total_t:,}")
print(f"  Mazos: {total_d:,}")
print(f"  Cartas únicas: {total_c:,}")
print(f"  Mazos con cartas: {scraped:,}")


# Checkpoint WAL + verificación de integridad antes de descargar
print("Aplicando WAL checkpoint...")
conn.execute("PRAGMA wal_checkpoint(TRUNCATE)")
conn.commit()

ic = conn.execute("PRAGMA integrity_check").fetchone()[0]
if ic == "ok":
    print("✅ integrity_check: ok — el archivo está listo para descargar")
else:
    print(f"⚠️  integrity_check: {ic[:200]}")
    print("   Revisá el scraping antes de descargar")
conn.close()

try:
    from google.colab import files
    files.download(DB_FILE)
except ImportError:
    print(f"Archivo en: {DB_FILE}")